In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, MaxPooling2D, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
# Accessing the folder on google drive
from google.colab import drive
drive.mount('/content/drive')
boat_data = '/content/drive/MyDrive/ML_Assignment/FishImgDataset'
train_data = '/content/drive/MyDrive/ML_Assignment/FishImgDataset/train'
test_data = '/content/drive/MyDrive/ML_Assignment/FishImgDataset/test'
val_data = '/content/drive/MyDrive/ML_Assignment/FishImgDataset/val'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Ensuring all image have the same size
batch_size = 100
img_heigth = 180
img_width = 180
epochs = 130
Num_Catergories = 31

In [4]:
# using the Imagedatagenerator to generalize the training images
train_datagen = ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 20,
    horizontal_flip = True,
    width_shift_range = 0.2,
    height_shift_range = 0.2,
    shear_range = 0.2,
    zoom_range = 0.2,
    fill_mode = 'nearest'
)

In [5]:
# using the ImageDataGenerator to generalize the test data
test_datagen = ImageDataGenerator(
    rescale = 1./255
)

In [6]:
val_datagen = ImageDataGenerator(
    rescale = 1./255
)

In [7]:
val_generator = val_datagen.flow_from_directory(
    val_data,
    target_size = (img_heigth, img_width),
    batch_size = batch_size,
    class_mode = 'categorical',
    shuffle = False,
)

Found 2751 images belonging to 31 classes.


In [8]:
# preparing the training data
train_generator = train_datagen.flow_from_directory(
    train_data,
    target_size = (img_heigth, img_width),
    batch_size = batch_size,
    class_mode = 'categorical',
)

Found 8779 images belonging to 31 classes.


In [9]:
#preparin the test data
test_generator = test_datagen.flow_from_directory(
    test_data,
    target_size = (img_heigth, img_width),
    batch_size = batch_size,
    class_mode = 'categorical',
    shuffle = False,
)

Found 1760 images belonging to 31 classes.


In [10]:
# creating a CNN model
model = Sequential([
    #input layer
    Conv2D(32, (3,3), padding='same', activation='relu', input_shape = (img_heigth,img_width,3)),
    MaxPooling2D(pool_size=(2,2)),

    #hidden layers
    Conv2D(64, (3,3), padding='same', activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    Conv2D(128, (3,3), padding = 'same', activation = 'relu'),
    MaxPooling2D(pool_size=(2,2)),

    Flatten(),
    Dropout(0.5),
    Dense(128, activation='relu'),

    #output layer
    Dense(Num_Catergories, activation='softmax')])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
# model copilation
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001),
    loss =  'categorical_crossentropy',
    metrics = ['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 180, 180, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 90, 90, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 90, 90, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 45, 45, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 45, 45, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 22, 22, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 61952)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 61952)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     7,929,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 31)             │         3,999 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,027,231 (30.62 MB)

 Trainable params: 8,027,231 (30.62 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# to train the model
STEPS_PER_EPOCH = train_generator.samples // batch_size
VALIDATION_STEPS = val_generator.samples // batch_size
history = model.fit(
    train_generator,
    epochs= epochs,
    validation_data=val_generator,
    validation_steps=VALIDATION_STEPS
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/130
10/88 ━━━━━━━━━━━━━━━━━━━━ 23:16 18s/step - accuracy: 0.0500 - loss: 3.7106

In [ ]:
# to evaluate or test the model on a test data
TEST_STEPS = test_generator.samples // batch_size

evaluation = model.evaluate(
    test_generator,
    steps = TEST_STEPS,
    verbose = 1
)
test_loss = test_results[0]
test_accuracy = test_results[1]

print(f"\n Final Test Loss: {test_loss:.4f}")
print(f"Final Test Accuracy: {test_accuracy:.4f}")


In [ ]:
# perdiction
import numpy as np
from tensorflow.keras.models import load_model

from tensorflow.keras.preprocessing import image

def predict_single_image(img_path, model, image_size):
    """
    Loads an image, preprocesses it, and makes a prediction using the model.
    """
    # Load the image and resize it
    img = image.load_img(img_path, target_size=image_size)

    #Convert the image to a NumPy array
    img_array = image.img_to_array(img)

    #Rescale pixel values (0-255 -> 0-1)
    img_array = img_array / 255.0

    #Expand dimensions to create a batch of size 1 (required by the model)
    # Shape changes from (150, 150, 3) to (1, 150, 150, 3)
    img_array = np.expand_dims(img_array, axis=0)

    #Make the prediction
    # This returns an array of probabilities, e.g., [[0.9, 0.05, 0.05]]
    predictions = model.predict(img_array)
    return predictions

# We reuse the size defined earlier
IMAGE_SIZE = (img_heigth, img_width)

In [ ]:
# Excute the prediction
#getting the indices of the classification
class_indices = train_generator.class_indices

# Invert the dictionary to map index (0, 1, 2) back to class name ("tilapia", etc.)
idx_to_class = {v: k for k, v in class_indices.items()}

# loading the perdiction image
img_perdict =  '/content/drive/MyDrive/ML_Assignment/FishImgDataset/test/Goby/111443-0mm.jpg'

# Make the prediction
raw_predictions = predict_single_image(img_perdict, model, IMAGE_SIZE)

# Get the predicted class index
predicted_index = np.argmax(raw_predictions, axis=1)[0]

# Get the probability for the predicted class
confidence = raw_predictions[0][predicted_index] * 100

# Get the predicted class name
predicted_class_name = idx_to_class[predicted_index]

print(f"Predicted Class: **{predicted_class_name}**")
print(f"Confidence: **{confidence:.2f}%**")

In [ ]:
# vizualization of a confusion matrix to see the accuray and the losses of the model
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

test_generator.shuffle = False
predictions = model.predict(test_generator)

# Convert probabilities to class labels
predicted_classes = np.argmax(predictions, axis=1)

# Get the true labels from the test generator
true_classes = test_generator.classes

# Get the class names (labels) for plot labels
class_labels = list(test_generator.class_indices.keys())

# Calculate Confusion Matrix
cm = confusion_matrix(true_classes, predicted_classes)

# 4. Plot the Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels
)
plt.title('Confusion Matrix for Fish Classification')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()



